### 11. El Costo del Marketing Inútil (Agregación con Filtro): Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida

In [0]:
%sql
SELECT
    COUNT (*) AS reservas_tato_canceladas,
    COUNT (*) * 1.5 AS perdida_marketing_usd
FROM cursodataacademy.hotel_db.tb_reservas
WHERE distribution_channel = 'TA/TO' AND is_canceled = 1;

--CONCLUSION: Se pagaron USD 60097.50 a las agencias debido a la cancelacion de 40065 reservas.

### 12. El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
%sql
--Se crea una tabla temporal para manipular las fechas de llegada y creacion de reserva.
CREATE OR REPLACE TEMP VIEW tv_reservas AS
SELECT
    *,
    TO_DATE(
    CONCAT_WS('-', arrival_date_year, arrival_date_month, arrival_date_day_of_month), 'yyyy-MMMM-d'
    ) AS arrival_date
FROM cursodataacademy.hotel_db.tb_reservas;

In [0]:
%sql
SELECT
    COUNT (*) AS cancelaciones_tardias,
    COUNT (*) * 120 AS perdida_operativa_usd
FROM tv_reservas
WHERE is_canceled = 1
AND DATE_DIFF(
    arrival_date,
    TO_DATE(reservation_status_date, 'd/M/yy')) < 3;

--CONCLUSION: Hubieron 3085 cancelaciones tardias generando una perdida operativa de USD 370200.

### 13. (JOIN 1 - LEFT JOIN - Entendiendo el Modelo): Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

In [0]:
%sql
-- Se crea una tabla temporal para relacionar los codigos de paises con su nombre.
CREATE OR REPLACE TEMP VIEW tb_paises AS
SELECT * FROM VALUES
    ('ABW', 'Aruba'),   ('AGO', 'Angola'),  ('AIA', 'Anguila'), ('ALB', 'Albania'),
    ('AND', 'Andorra'), ('ARG', 'Argentina'),   ('ARM', 'Armenia'), ('ASM', 'Samoa Americana'),
    ('ATA', 'Antártida'),   ('AUS', 'Australia'),   ('AUT', 'Austria'), ('AZE', 'Azerbaiyán'),
    ('BDI', 'Burundi'), ('BEL', 'Bélgica'), ('BEN', 'Benín'),   ('BGD', 'Bangladés')
AS (cod_pais, nombre_pais);

In [0]:
%sql
SELECT
    r.is_canceled,
    p.nombre_pais
FROM cursodataacademy.hotel_db.tb_reservas r
LEFT JOIN tb_paises p
ON r.country = p.cod_pais;

--CONCLUSION: Relacionamos las reservas con el nombre del pais de origen de la reserva.

### 14. (JOIN 2 - INNER JOIN con Agrupación): Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?

In [0]:
%sql
SELECT
    p.nombre_pais,
    COUNT (*) AS reservas_canceladas
FROM cursodataacademy.hotel_db.tb_reservas r
INNER JOIN tb_paises p
    ON r.country = p.cod_pais
WHERE is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY reservas_canceladas DESC
LIMIT 5;

--CONCLUSION: Los cinco paises que poseen mayor volumen de reservas canceladas fueron Bélgica, Austria, 
--            Angola, Australia y Argentina. En este caso Bélgica termino con un total de 472 cancelaciones.

### 15. (JOIN 3 - Simulación FULL OUTER JOIN): Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.

In [0]:
%sql
-- Se crean dos tablas temporales para relacionar las reservas con el vendedor.
CREATE OR REPLACE TEMP VIEW tb_reservas_vendedor AS
SELECT * , 
    TRY_CAST(agent AS INT) AS id_vendedor
FROM cursodataacademy.hotel_db.tb_reservas;

CREATE OR REPLACE TEMP VIEW tb_vendedores AS
SELECT DISTINCT 
    id_vendedor, 
    CONCAT('Vendedor ', 
    CAST(id_vendedor AS STRING)) AS nombre_vendedor
FROM tb_reservas_vendedor
WHERE id_vendedor IS NOT NULL
UNION ALL
SELECT * 
FROM VALUES 
    (9001, 'Vendedor 9001'), 
    (9002, 'Vendedor 9002') AS (id_vendedor, nombre_vendedor);

In [0]:
%sql
SELECT
    rv.distribution_channel,
    v.nombre_vendedor
FROM tb_reservas_vendedor rv
FULL OUTER JOIN tb_vendedores v
    ON rv.id_vendedor = v.id_vendedor;
--CONCLUSION: El cruce nos permite visualizar las reservas sin un vendedor coincidente como los vendedores  --            sin reservas, esto nos permite revisar las asignaciones comerciales de cada uno.

### 16. (JOIN 4 - RIGHT JOIN): El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).

In [0]:
%sql
--Se crean dos tablas temporales para relaciónar las reservas con el monto facturado
CREATE OR REPLACE TEMP VIEW tb_reservas_id AS
SELECT
    *,
    ROW_NUMBER() OVER 
    (
    ORDER BY hotel, arrival_date_year, arrival_date_month, arrival_date_day_of_month,
             lead_time, adr, stays_in_week_nights, stays_in_weekend_nights, adults, is_canceled, country, agent
    ) AS id_reserva
FROM cursodataacademy.hotel_db.tb_reservas;

CREATE OR REPLACE TEMP VIEW tb_facturas AS
SELECT * FROM VALUES
    (1, 320.00),  (3, 415.50),  (5, 280.00),  (10, 505.25),  (20, 190.00),
    (25, 610.00),  (28, 275.75),  (32, 340.00)
AS (id_reserva, monto_facturado);

In [0]:
%sql
SELECT
    f.id_reserva,
    f.monto_facturado,
    r.hotel
FROM tb_reservas_id r
RIGHT JOIN tb_facturas f
    ON r.id_reserva = f.id_reserva;

--CONCLUSION: Notamos que el id_reserva 25 tiene un mayor monto facturado, ademas se conservaron todas las facturas emitidas, facilitando la revisión de 
--            diferencias entre faturación y reservas.